In [1]:
import argparse
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
matplotlib.use('Agg')
import joblib

from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from xgboost import XGBRegressor


In [2]:
# Configuration
# --------------------------------------------------------------------------- #

RAW_FEATURES = [
    "irradiance_wm2",
    "rainfall_mm",
    "relative_humidity_pct",
    "sea_level_pressure_hpa",
    "temperature_c",
    "visibility_km",
    "wind_speed_ms",
]

TIME_FEATURES = ["hour", "minute", "day", "month", "year"]

# Cyclical encodings derived from TIME_FEATURES (see feature engineering step)
CYCLICAL_FEATURES = ["hour_sin", "hour_cos", "month_sin", "month_cos"]

# Station/site identifier — critical for capturing per-site capacity/calibration
SITE_FEATURE = "site_id_ttl"

ALL_FEATURES = RAW_FEATURES + CYCLICAL_FEATURES + ["day"] + [SITE_FEATURE]

TARGET_COL = "normalized_generation"
FUTURE_TARGET_COL = "target_1h"

# 15-minute native sampling interval; 1 hour ahead = 4 steps.
INTERVAL = pd.Timedelta(minutes=15)
FORECAST_STEPS = 4
FORECAST_DELTA = INTERVAL * FORECAST_STEPS  # exactly 1 hour

TRAIN_FRACTION = 0.8

OUTPUT_GRAPH_DIR = "output_graph"
MODELS_DIR = "models"
METRICS_DIR = "metrics"

In [3]:
# 1. Load
df = pd.read_csv('data/stage-3/training_data.csv')
test_df = pd.read_csv('data/stage-3/test_data.csv')

print(df.columns.tolist())
print(test_df.columns.tolist())

['hour', 'minute', 'day', 'month', 'year', 'site_id_ttl', 'irradiance_wm2', 'rainfall_mm', 'relative_humidity_pct', 'sea_level_pressure_hpa', 'temperature_c', 'visibility_km', 'wind_speed_ms', 'normalized_generation']
['hour', 'minute', 'day', 'month', 'year', 'site_id_ttl', 'irradiance_wm2', 'rainfall_mm', 'relative_humidity_pct', 'sea_level_pressure_hpa', 'temperature_c', 'visibility_km', 'wind_speed_ms', 'normalized_generation']


In [4]:
for d in [df, test_df]:
    d['hour_sin'] = np.sin(2 * np.pi * d['hour'] / 24)
    d['hour_cos'] = np.cos(2 * np.pi * d['hour'] / 24)
    d['month_sin'] = np.sin(2 * np.pi * d['month'] / 12)
    d['month_cos'] = np.cos(2 * np.pi * d['month'] / 12)

# Align site_id_ttl as a shared categorical dtype across train and test
all_sites = pd.concat([df['site_id_ttl'], test_df['site_id_ttl']]).unique()
df['site_id_ttl'] = pd.Categorical(df['site_id_ttl'], categories=all_sites)
test_df['site_id_ttl'] = pd.Categorical(test_df['site_id_ttl'], categories=all_sites)

In [5]:

df.corr(numeric_only = True)

,hour,minute,day,month,year,irradiance_wm2,rainfall_mm,relative_humidity_pct,sea_level_pressure_hpa,temperature_c,visibility_km,wind_speed_ms,normalized_generation,hour_sin,hour_cos,month_sin,month_cos
hour,1.000000,-0.000185,0.000713,-0.001311,0.000504,0.029488,-0.011463,-0.091115,-0.027309,0.065377,0.061784,-0.022708,0.024071,-0.775827,-0.101584,0.000377,0.000243
minute,-0.000185,1.000000,-0.000100,-0.000222,-0.000026,0.003029,-0.001541,-0.000326,0.000231,0.000366,0.000146,0.000855,-0.000356,0.000373,0.000247,0.000052,-0.000196
day,0.000713,-0.000100,1.000000,0.017218,-0.000485,-0.021597,-0.002739,0.042830,0.035525,-0.003799,-0.065404,-0.093153,-0.019795,-0.000886,0.000068,-0.008779,0.003515
month,-0.001311,-0.000222,0.017218,1.000000,-0.375985,0.037167,-0.013666,-0.281196,0.017328,0.158230,0.160329,0.190963,0.010627,0.001744,0.000321,-0.696626,0.371944
year,0.000504,-0.000026,-0.000485,-0.375985,1.000000,0.010133,-0.013309,0.005973,0.168616,-0.199192,-0.121042,0.003626,-0.011414,-0.000332,-0.000422,0.387468,-0.006012
irradiance_wm2,0.029488,0.003029,-0.021597,0.037167,0.010133,1.000000,-0.046297,-0.492864,-0.081532,0.416731,0.200028,0.042290,0.948255,0.027352,-0.693933,-0.079119,-0.057829
rainfall_mm,-0.011463,-0.001541,-0.002739,-0.013666,-0.013309,-0.046297,1.000000,0.096845,-0.064454,-0.001950,-0.301003,0.040777,-0.046598,0.012789,-0.022639,-0.009122,-0.057615
relative_humidity_pct,-0.091115,-0.000326,0.042830,-0.281196,0.005973,-0.492864,0.096845,1.000000,-0.239693,-0.054135,-0.401622,-0.193645,-0.479466,0.088217,0.332269,0.088169,-0.323924
sea_level_pressure_hpa,-0.027309,0.000231,0.035525,0.017328,0.168616,-0.081532,-0.064454,-0.239693,1.000000,-0.797379,-0.005685,0.169400,-0.088724,0.077930,0.042016,0.411451,0.720290
temperature_c,0.065377,0.000366,-0.003799,0.158230,-0.199192,0.416731,-0.001950,-0.054135,-0.797379,1.000000,0.179167,-0.255535,0.420811,-0.066865,-0.241574,-0.569764,-0.595783


In [6]:
features = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day',
    'irradiance_wm2', 'rainfall_mm', 'relative_humidity_pct',
    'sea_level_pressure_hpa', 'temperature_c', 'visibility_km', 'wind_speed_ms',
    'site_id_ttl'
]
target = 'normalized_generation'

# Ensure datetime exists on both frames before splitting
df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour', 'minute']])
test_df['datetime'] = pd.to_datetime(test_df[['year', 'month', 'day', 'hour', 'minute']])

X_train, y_train = df[features], df[target]
X_test, y_test = test_df[features], test_df[target]

# Keep datetime aligned with X_test/y_test by index, for plotting later
dt_test = test_df['datetime']

# Use last ~3 months of 2022 as validation, rest as training
val_mask = (df['year'] == 2022) & (df['month'] >= 10)

X_tr, y_tr = df.loc[~val_mask, features], df.loc[~val_mask, target]
X_val, y_val = df.loc[val_mask, features], df.loc[val_mask, target]

# Keep datetime aligned with train/val splits too, for any future plotting
dt_tr = df.loc[~val_mask, 'datetime']
dt_val = df.loc[val_mask, 'datetime']

print(X_tr.dtypes['site_id_ttl'])  # should show 'category'

category


In [7]:
# --- Feature set: include site_id_ttl (fixes per-site capacity/calibration issue) ---
features = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'day',
    'irradiance_wm2', 'rainfall_mm', 'relative_humidity_pct',
    'sea_level_pressure_hpa', 'temperature_c', 'visibility_km', 'wind_speed_ms',
    'site_id_ttl'
]
target = 'normalized_generation'

X_tr, y_tr = df.loc[~val_mask, features], df.loc[~val_mask, target]
X_val, y_val = df.loc[val_mask, features], df.loc[val_mask, target]
X_test, y_test = test_df[features], test_df[target]

# --- Main model ---
model = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.02,          # smaller steps, avoids premature early stopping (best_iteration was only 59 at lr=0.05)
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    early_stopping_rounds=150,   # more patience before stopping
    enable_categorical=True,     # required since site_id_ttl is a category dtype
    random_state=42
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_tr, y_tr), (X_val, y_val)],
    verbose=False,
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",150
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [8]:

print("Best iteration:", model.best_iteration)

# --- Evaluate on true 2023 test set ---
y_pred = model.predict(X_test)

print("RMSE:", root_mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))

Best iteration: 291
RMSE: 0.02524594832428075
MAE: 0.014126855457437606
R²: 0.815073523015337


In [88]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

y_pred3 = model.predict(X_test)

# Pick a random day from the available dates in test_df
available_days = test_df['datetime'].dt.date.unique()
random_day = np.random.choice(available_days)
random_day = pd.Timestamp(random_day)

mask = (test_df['datetime'] >= random_day) & (test_df['datetime'] < random_day + pd.Timedelta(days=2))
times = test_df.loc[mask, 'datetime']

actual_vals = y_test[mask.values]
pred_vals = y_pred3[mask.values]

peak_actual = actual_vals.max()
peak_pred = pred_vals.max()

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(times, actual_vals, color='black', linewidth=1.5, label='Actual')
ax.plot(times, pred_vals, color='#2ecc71', linewidth=1.5, label='Model 3')

ax.set_title('Actual vs Model 3 — All Sites', fontsize=13)
ax.set_xlabel('Time')
ax.set_ylabel('normalized_generation')
ax.legend(loc='upper right', frameon=True)
ax.grid(alpha=0.3)

# Date + peak power level label
label_text = (
    f"{random_day.strftime('%B %d, %Y')}\n"
    f"Peak Actual: {peak_actual:.3f} kwh\n"
    f"Peak Predicted: {peak_pred:.3f} kwh"
)

ax.text(
    0.02, 0.95, label_text,
    transform=ax.transAxes,
    fontsize=12, fontweight='bold',
    verticalalignment='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.9)
)

plt.tight_layout()
plt.savefig('actual_vs_model3_random_day.png', dpi=150, bbox_inches='tight', facecolor='white')
print(f"Plotted random day: {random_day.strftime('%Y-%m-%d')}")
print("Saved to actual_vs_model3_random_day.png")

Plotted random day: 2023-08-03
Saved to actual_vs_model3_random_day.png


In [ ]:
model.save_model("urja-latest.json")

In [95]:
import joblib

# Save
joblib.dump(model, "urja.joblib")

['urja.joblib']